# Omniprésence et anthropomorphisation des Grands Modèles de Langue dans la recherche en TAL : étude de cas exploratoire sur un corpus tiré de la conférence TALN

In [ ]:
import pandas as pd
import pickle
import spacy
import os
from spacy import displacy
from utils import return_LLM_entities_ids, return_LLM_entities_ids_en
from tqdm import tqdm
from acl_anthology import Anthology
import re

## Extraction des articles JEP-TALN-RECITAL depuis l'ACL Anthology

Dans un premier temps, on récupère les méta-données des articles JEP-TALN-RECITAL contenus dans l'ACL Anthology via la librairie [acl_anthology](https://acl-anthology.readthedocs.io/latest/guide/getting-started/).

In [2]:
# def get_year(paper):
#     try:
#         return int(re.search(re.compile(r"(19\d{2}|20[01]\d|202[0-5])"), paper.bibkey).group())
#     except:
#         return -1

# def get_paper_events(paper):
#     return [event.id for event in paper.get_events()]
    
# def get_papers_in_event(papers, event_key):
#     return [p for p in papers if event_key in get_paper_events(p)]

In [3]:
# # load the anthology
# anthology = Anthology.from_repo()

# # store all papers 
# all_papers = list(anthology.papers())
# print(len(all_papers))

# # get only JEP-TALN-RECITAL papers of years 2020-2025
# jeptalnrecital_events = ["jeptalnrecital-" + str(year) for year in range(2020, 2026)]
# jeptalnrecital_papers = []

# for event in jeptalnrecital_events:
#     jeptalnrecital_papers.extend(get_papers_in_event(all_papers, event))

# # exclude the proceeding documents to keep only articles
# for p in jeptalnrecital_papers:

#     if str(p.title).startswith("Actes de"):
#         jeptalnrecital_papers.remove(p)

    
# print(len(jeptalnrecital_papers))

On stocke les informations sur les articles identifiés dans un DataFrame pandas.

In [4]:
# data = []

# for p in jeptalnrecital_papers:
    
#     row = [
#         p.bibkey,
#         p.id,
#         p.full_id,
#         str(p.title),
#         str(p.abstract),
#         get_paper_events(p),
#         get_year(p)
#     ]

#     data.append(row)

# df = pd.DataFrame(data, columns = ["bibkey", "id", "full_id", "title", "abstract", "events", "year"])

In [5]:
data_folder = "data/"
jeptalnrecital_file_name = "jeptalnrecital-papers-VF"

# with open(os.path.join(data_folder, jeptalnrecital_file_name) + ".pkl", "wb") as f:
#     pickle.dump(df, f)

with open(os.path.join(data_folder, jeptalnrecital_file_name) + ".pkl", "rb") as f:
    df = pickle.load(f).sort_values(by = "bibkey").reset_index(drop = True)

In [6]:
df.head()

,bibkey,id,full_id,title,abstract,events,year
0,aabid-etal-2025-quand,15,2025.jeptalnrecital-coria.15,Quand les Bots Déjouent l’Apprentissage : Enje...,Identifier les bots d’une une bibliothèque num...,[jeptalnrecital-2025],2025
1,abchiche-etal-2023-integration,11,2023.jeptalnrecital-coria.11,Intégration du raisonnement numérique dans les...,"Ces dernières années, les modèles de langue on...",[jeptalnrecital-2023],2023
2,abdoulaye-ngom-etal-2025-analyse,1,2025.jeptalnrecital-coria.1,Analyse Textuelle et Extraction Géospatiale po...,L’Afrique de l’Ouest fait face à une insécurit...,[jeptalnrecital-2025],2025
3,abdul-rauf-etal-2020-simplification,33,2020.jeptalnrecital-taln.33,Simplification automatique de texte dans un co...,La simplification de textes a émergé comme un ...,[jeptalnrecital-2020],2020
4,abel-etal-2024-synthese,19,2024.jeptalnrecital-jep.19,Synthèse de gestes communicatifs via STARGATE,La synthèse de gestes lié à la parole est un d...,[jeptalnrecital-2024],2024


## Analyse spaCy et identification des mentions de LLM

Le titre et l'abstract de chaque article peuvent être identifiés par un modèle spaCy. Ceci nous permet d'exploiter les formes lemmatisées des mots ainsi que des dépendances syntaxiques (enfants) pour identifier les mentions de LLM (en complément de mots-clés).  

On utilise le modèle [https://spacy.io/models/fr#fr_dep_news_trf](https://spacy.io/models/fr#fr_dep_news_trf).

In [7]:
# model = spacy.load("fr_dep_news_trf")

In [8]:
# llm_papers_ids = []
# llm_entities_ids = []
# llm_entities = []
# docs = []

# LLM_names = ["llama", "qwen", "bert", "deepseek", "mistral", "gpt", "gemma", "gemini", "bard", "claude"]
# LLM_entities = ["ia", "llm", "llms"]

# for i, row in tqdm(df.iterrows(), total = df.shape[0]):
    
#     title = row["title"]
#     abstract = row["abstract"]

#     # concaténation titre et abstract
#     title_and_abstract = title.strip(".") + ".\n" + abstract

#     # analyse du titre et abstract, stockage des résultats pour analyses ultérieures
#     doc = model(title_and_abstract)
#     docs.append(doc)

#     # on récupère les entités dénotant des LLM pour chaque article
#     entities_ids = return_LLM_entities_ids(doc, LLM_entities, LLM_names)
#     llm_entities_ids.append(entities_ids)

#     entities = [tok.orth_ for j, tok in enumerate(doc) if j in entities_ids]
#     llm_entities.append(entities)
    
#     if len(entities_ids) > 0:
#         llm_papers_ids.append(row["full_id"])

# print(len(llm_papers_ids))

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 737/737 [03:44<00:00,  3.28it/s]

271


In [9]:
# df["analyzed_text"] = docs
# df["llm_entities_ids"] = llm_entities_ids
# df["llm_entities"] = df.apply(lambda x: [tok.orth_ for i, tok in enumerate(x["analyzed_text"]) if i in x["llm_entities_ids"]], axis = 1)

In [10]:
df[df["llm_entities"].apply(lambda x : x!= [])]

,bibkey,id,full_id,title,abstract,events,year,analyzed_text,llm_entities_ids,llm_entities
1,abchiche-etal-2023-integration,11,2023.jeptalnrecital-coria.11,Intégration du raisonnement numérique dans les...,"Ces dernières années, les modèles de langue on...",[jeptalnrecital-2023],2023,"(Intégration, du, raisonnement, numérique, dan...","[6, 25, 62, 78, 92, 115, 171, 192]","[modèles, modèles, modèles, modèles, modèles, ..."
2,abdoulaye-ngom-etal-2025-analyse,1,2025.jeptalnrecital-coria.1,Analyse Textuelle et Extraction Géospatiale po...,L’Afrique de l’Ouest fait face à une insécurit...,[jeptalnrecital-2025],2025,"(Analyse, Textuelle, et, Extraction, Géospatia...","[105, 114]","[modèles, CamemBERT]"
4,abel-etal-2024-synthese,19,2024.jeptalnrecital-jep.19,Synthèse de gestes communicatifs via STARGATE,La synthèse de gestes lié à la parole est un d...,[jeptalnrecital-2024],2024,"(Synthèse, de, gestes, communicatifs, via, STA...","[45, 127]","[agents, modèle]"
10,adam-cuvillier-etal-2024-les,9,2024.jeptalnrecital-taln.9,Les représentations contextuelles stéréotypées...,Nous présentons une étude pour mieux identifie...,[jeptalnrecital-2024],2024,"(Les, représentations, contextuelles, stéréoty...","[6, 35, 105, 150]","[modèles, modèles, modèles, modèles]"
13,aguiar-etal-2025-inference,18,2025.jeptalnrecital-trad.18,Inférence en langue naturelle appliquée au rec...,Recruter des patients pour les essais clinique...,[jeptalnrecital-2025],2025,"(Inférence, en, langue, naturelle, appliquée, ...","[133, 210, 259]","[modèle, modèles, modèle]"
...,...,...,...,...,...,...,...,...,...,...
723,zanella-baril-2025-la,28,2025.jeptalnrecital-taln.28,La confiance de Mistral-7B est-elle justifiée ...,Évaluer la fiabilité des grands modèles de lan...,[jeptalnrecital-2025],2025,"(La, confiance, de, Mistral-7B, est-elle, just...","[3, 22, 26, 56, 94, 178, 209, 230]","[Mistral-7B, modèles, LLMs, Mistral-7B, Mistra..."
732,zhou-etal-2025-explicabilite,1,2025.jeptalnrecital-diagllm.1,Explicabilité par Perturbations pour les Systè...,Les systèmes de Génération Augmentée par Récup...,[jeptalnrecital-2025],2025,"(Explicabilité, par, Perturbations, pour, les,...","[26, 30, 162]","[Modèles, LLM, modèles]"
733,zhu-etal-2022-flux,2,2022.jeptalnrecital-humanum.2,Flux d’informations dans les systèmes encodeur...,Ce travail présente deux séries d’expériences ...,[jeptalnrecital-2022],2022,"(Flux, d’, informations, dans, les, systèmes, ...","[80, 86]","[modèle, modèle]"
734,zong-piwowarski-2023-xpmir,19,2023.jeptalnrecital-coria.19,XPMIR: Une bibliothèque modulaire pour l’appre...,"Ces dernières années, plusieurs librairies pou...",[jeptalnrecital-2023],2023,"(XPMIR, :, Une, bibliothèque, modulaire, pour,...","[82, 100, 116, 165]","[modèles, modèles, modèles, modèles]"


Inspection partielle des entités identifiées comme dénotant des LLM, car des faux positifs sont possibles, par ex à cause de la chaîne  "bert" qui peut se retrouver dans des noms propres.

In [11]:
# vérification des entités LLM extraites
all_llm_entities = []

for i, row in df.iterrows():
    for e in row["llm_entities"]:
        all_llm_entities.append(e.lower())

print(sorted(list(set(all_llm_entities))))

['adminbert', 'agent', 'agents', 'albert', 'bert', 'bertalign', 'bertrand', 'bertscore', 'biobert', 'bluebert', 'camembert', 'chatgpt', 'chatgpt-4o', 'claude', 'colbert', 'deepseek', 'distilbert', 'drbert', 'etqwen2.5', 'flaubert', 'gemini', 'gemma', 'gpt', 'gpt-2', 'gpt-3', 'gpt-3.5', 'gpt-4', 'gpt-4o', 'gptscore', 'https://huggingface.co/jgmorenof/mistral_merged_0_4', 'hubert', 'ia', 'intelligence', 'intelligences', 'llama', 'llama-3', 'llama-3.3', 'llama2', 'llama3', 'llama3.0', 'llama3.1', 'llm', 'llms', 'mbert', 'mistral', 'mistral-7b', 'modèle', 'modèles', 'ollama', 'pubmedbert', 'qwen', 'roberta', 'scibert', 'selfcheckgpt']


In [12]:
from collections import Counter

c = Counter(all_llm_entities)
c

Counter({'modèles': 491,
         'modèle': 190,
         'llm': 100,
         'llms': 73,
         'bert': 62,
         'camembert': 34,
         'ia': 25,
         'agent': 16,
         'chatgpt': 15,
         'intelligence': 12,
         'agents': 8,
         'gpt': 7,
         'llama': 6,
         'flaubert': 5,
         'gpt-4o': 4,
         'hubert': 4,
         'mistral-7b': 4,
         'adminbert': 4,
         'drbert': 3,
         'gpt-2': 3,
         'qwen': 3,
         'scibert': 2,
         'roberta': 2,
         'mistral': 2,
         'claude': 2,
         'gpt-4': 2,
         'albert': 2,
         'llama3': 2,
         'llama-3': 2,
         'mbert': 2,
         'gemini': 2,
         'colbert': 2,
         'bertrand': 1,
         'biobert': 1,
         'bluebert': 1,
         'gpt-3.5': 1,
         'https://huggingface.co/jgmorenof/mistral_merged_0_4': 1,
         'bertscore': 1,
         'bertalign': 1,
         'distilbert': 1,
         'gpt-3': 1,
         'llama2': 1,

In [18]:
entities_to_check = ["bertrand", "bertscore", "claude", "https://huggingface.co/jgmorenof/mistral_merged_0_4", "hubert"]

for i, row in df[df["llm_entities"].apply(lambda x: len(set([elt.lower() for elt in x]).intersection(set(entities_to_check))) > 0)].iterrows():
    print(i)
    print(row["llm_entities_ids"])
    print(row["llm_entities"])
    print(row["analyzed_text"])
    print()

117
[2, 67, 75, 78, 140, 158]
['modèles', 'modèles', 'modèles', 'HuBERT', 'modèle', 'modèles']
Adaptation de modèles auto-supervisés pour la reconnaissance de phonèmes dans la parole d’enfant.
La reconnaissance de parole d’enfant est un domaine de recherche encore peu développé en raison du manque de données et des difficultés caractéristiques de cette tâche. Après avoir exploré diverses architectures pour la RAP d’enfant dans de précédents travaux, nous nous attaquons dans cet article aux nouveaux modèles auto-supervisés. Nous comparons d’abord plusieurs modèles Wav2vec2, HuBERT et WavLM adaptés superficiellement à la reconnaissance de phonèmes sur parole d’enfant, et poursuivons nos expériences avec le meilleur d’entre eux, un WavLM base+. Il est ensuite adapté plus profondément en dégelant ses blocs transformer lors de l’entraînement sur parole d’enfant, ce qui améliore grandement ses performances et le fait surpasser significativement notre modèle de base, un Transformer+CTC. Enfin

Correction manuelle dans le DataFrame des cas où il s'agit clairement de faux positifs :

In [23]:
# # Faux positif: 'Hubert' dans 'la parole d'Hubert Pernot'
# df.at[168, "llm_entities_ids"] = []
# df.at[168, "llm_entities"] = []

# # Faux positif: 'Betrand' dans '(Bertrand et al., 2008)'
# df.at[185, "llm_entities_ids"] = []
# df.at[185, "llm_entities"] = []

# # Faux positif: lien vers un modèle Mistral mais pas mention directe du modèle
# df.at[286, "llm_entities_ids"] = [5, 28, 33, 51, 130, 135, 171, 187, 212, 248, 260, 299]
# df.at[286, "llm_entities"] = ['modèles', 'modèles', 'LLM', 'modèles', 'modèles', 'LLM', 'LLM', 'modèle', 'LLM', 'modèles', 'modèles', 'modèle']

# # Faux positif: BERTScore est une mention de métrique et non pas de modèle
# # Remarque: 'GML' comme faux négatif, à intégrer dans future liste de mots-clés, pas ajouté ici
# df.at[376, "llm_entities_ids"] = []
# df.at[376, "llm_entities"] = []

# # Faux positif: 'Hubert' dans 'Lab Hubert Curien'
# df.at[290, "llm_entities_ids"] = []
# df.at[290, "llm_entities"] = []

In [24]:
df.loc[[168, 185, 286, 376, 290]]

,bibkey,id,full_id,title,abstract,events,year,analyzed_text,llm_entities_ids,llm_entities
168,cecelewski-etal-2024-etude,8,2024.jeptalnrecital-jep.8,Étude en temps réel de la fusion des /a/ ~ /ɑ/...,Cette étude explore la variation diachronique ...,[jeptalnrecital-2024],2024,"(Étude, en, temps, réel, de, la, fusion, des, ...",[],[]
185,chlebowski-ballier-2020-cest,12,2020.jeptalnrecital-jep.12,"C’est “mm-hm, oui” ou “mm-hm, non” ? Propositi...",Cet article se propose d’envisager l’existence...,[jeptalnrecital-2020],2020,"(C’, est, “, mm, -, hm, ,, oui, ”, ou, “, mm, ...",[],[]
286,g-moreno-etal-2025-patientdx,22,2025.jeptalnrecital-trad.22,PatientDx : Fusion des grands modèles de langu...,L’affinage des grands modèles de langue (abrég...,[jeptalnrecital-2025],2025,"(PatientDx, :, Fusion, des, grands, modèles, d...","[5, 28, 33, 51, 130, 135, 171, 187, 212, 248, ...","[modèles, modèles, LLM, modèles, modèles, LLM,..."
376,jourdan-etal-2025-identification,25,2025.jeptalnrecital-taln.25,Identification de mesures d’évaluation fiables...,L’évaluation de la révision des textes scienti...,[jeptalnrecital-2025],2025,"(Identification, de, mesures, d’, évaluation, ...",[],[]
290,gallienne-poibeau-2023-quelques,1,2023.jeptalnrecital-statement.1,Quelques observations sur la notion de biais d...,Cet article revient sur la notion de biais dan...,[jeptalnrecital-2023],2023,"(Quelques, observations, sur, la, notion, de, ...",[],[]


In [25]:
df[df["llm_entities"].apply(lambda x : x!= [])].shape[0]

267

In [26]:
processed_jeptalnrecital_file_path = "analyzed-jeptalnrecital-papers"

# with open(os.path.join(data_folder, processed_jeptalnrecital_file_path) + ".pkl", "wb") as f:
#     pickle.dump(df, f)

with open(os.path.join(data_folder, processed_jeptalnrecital_file_path) + ".pkl", "rb") as f:
    df = pickle.load(f)

## Extraction de structures syntaxiques  

Selon relations de dépendances syntaxiques UD ([https://universaldependencies.org/u/dep/index.html](https://universaldependencies.org/u/dep/index.html)).

In [30]:
def get_token_idx(token, doc):
    idx = -1
    for i, tok in enumerate(doc):
        if tok == token:
            idx = i
    return idx

### Entités-LLM en position de sujet verbal

_POS=VERB --nsubj--> E  

In [35]:
all_verbs_tokens = []

for i, row in tqdm(df.iterrows(), total = df.shape[0]):

    # accès à la position des entités LLM et au texte analysé
    llm_entities_ids = row["llm_entities_ids"]
    doc = row["analyzed_text"]
    verbs = []
    
    for e_id in llm_entities_ids:
        
        tok = doc[e_id]
        deprel = tok.dep_
        head = tok.head
        
        # relation 'nsubj' entre un verbe et l'entité-LLM
        if deprel == "nsubj" and head.pos_ == "VERB":

            head_id = get_token_idx(head, doc)
            verbs.append((e_id, head_id))
            verb = head.lemma_
            
            # relation 'expl' indique présence de pronoms réflexifs
            # par ex : 's'appuyer'
            head_expl_children = [child for child in head.children if child.dep_.startswith("expl")]
            if len(head_expl_children) > 0:
                verb = "(" +head_expl_children[0].orth_ + ") " + verb


            # relation 'xcomp' indique un complément verbal
            # par ex : 'pouvoir prédire'
            head_xcomp_children = [child for child in head.children if child.dep_ == "xcomp" and child.pos_ == "VERB"]
            if len(head_xcomp_children) > 0:
                verb = verb + " (" + head_xcomp_children[0].orth_ + ")"

            # contexte autour de l'occurrence
            print(tok, verb)
            print(doc[max(0, e_id-5) : min(e_id+20, len(doc))])
            print()

    all_verbs_tokens.append(verbs)

df["verb_subj"] = all_verbs_tokens

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 737/737 [00:00<00:00, 35575.22it/s]

modèles connaître
Ces dernières années, les modèles de langue ont connu une évolution galopante grâce à l’augmentation de la puissance de calcul qui a rendu

modèles parvenir (effectuer)
un paradigme courant, les modèles actuels ne parviennent pas à effectuer des calculs de manière satisfaisante. Pour y remédier, une solution est

agent recommander
objectif pédagogique donné. Un agent de renforcement, entraîné en environnement simulé, recommande des activités en maximisant la progression attendue tout en respectant

modèles permettre
locuteurs. Même si les modèles de langue les plus récents ont permis des progrès remarquables dans ce domaine, générer un résumé fidèle au

modèles améliorer
oice 16.1 démontrent que les modèles entraînés sur les clusters découverts améliorent les performances des groupes démographiques désavantagés tout en conservant des performances compétitives et

LLMs surpasser
Les résultats montrent que les LLMs surpassent largement les méthodes traditionnelles en te

### Entités-LLM en position d'argument/de complément verbal

_POS=VERB --{obj, iobj, obl:arg}--> E

In [34]:
all_verbs_as_obj = set()
all_verbs_as_obj_tokens = []

for i, row in tqdm(df.iterrows(), total = df.shape[0]):
    
    llm_entities_ids = row["llm_entities_ids"]
    doc = row["analyzed_text"]
    verb_tokens = []
    
    for e_id in llm_entities_ids:
        
        tok = doc[e_id]
        deprel = tok.dep_
        head = tok.head
        
        if deprel in ["obj", "iobj", "obl:arg"] and head.pos_ == "VERB":

            # pronom réflexif attaché au verbe 
            head_expl_children = [child for child in head.children if child.dep_.startswith("expl")]
            verb = head_expl_children[0].orth_ + " " + head.lemma_ if len(head_expl_children) > 0 else head.lemma_

            # préposition attachée à l'entité
            entity_case_children = [child for child in tok.children if child.dep_ == "case"]
            verb = verb + " " + entity_case_children[0].orth_ if len(entity_case_children) > 0 else verb

            print(verb, tok)
            all_verbs_as_obj.add(verb)
            
            print(doc[e_id-5:e_id+20])
            print()

            head_id = get_token_idx(head, doc)
            verb_tokens.append((e_id, head_id))
    
    all_verbs_as_obj_tokens.append(verb_tokens)

df["verb_compl"] = all_verbs_as_obj_tokens

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 737/737 [00:00<00:00, 28850.91it/s]

entraîner modèles
solution est d’entraîner les modèles de langue à utiliser des outils externes tels qu’une calculatrice ou un “runtime” de code python

évaluer modèles
presse. Nous évaluons deux modèles d’extraction d’entités spatiales, GLiNER et CamemBERT, en configuration zéro-shot et après ajustement (

se refléter dans modèles
stéréotypes se reflètent dans les modèles de langue français. Nous adaptons le jeu de données StereoSet à la langue française et suivons le même

permettre au modèle
permet tout de même au modèle de langue de déterminer l’égilibilité du patient pour l’essai clinique, nous construisons la tâche Natural Language

soumettre à modèles
des amorces à plusieurs grands modèles de langue, et obtenons un score F1 compris entre 56,6 et 71,8 avec le langage patient, contre

combiner CamemBERT
, une approche hybride combinant CamemBERT et des variables linguistiques et des modèles de langue génératifs (LLMs). Une analyse approfondie de ces

utiliser LLMs
approches généra

### Adjectifs et attributs du sujet modifiants une entité-LLM

In [36]:
all_adj = set()
all_adj_ids = []

for i, row in tqdm(df.iterrows(), total = df.shape[0]):
    
    llm_entities_ids = row["llm_entities_ids"]
    doc = row["analyzed_text"]
    adj_ids = []
    
    for e_id in llm_entities_ids:
        
        tok = doc[e_id]
        deprel = tok.dep_
        children = tok.children
        head = tok.head
        
        for child in children:
            
            # Adjectifs 
            if child.dep_ == "amod" and child.pos_ == "ADJ":
                all_adj.add(child.lemma_)
                print(tok, child.orth_)
                print(doc[max(0, e_id-5):min(len(doc), e_id+15)])
                print()
                adj_id = get_token_idx(child, doc)
                adj_ids.append((e_id, adj_id))

        # Attributs du sujet
        if deprel == "nsubj" and head.pos_ == "ADJ":
            all_adj.add(head.lemma_)
            print(tok, head.orth_, "(attr)")
            print(doc[max(0, e_id-5):min(len(doc), e_id+15)])
            print()

            attr_id = get_token_idx(head, doc)
            adj_ids.append((e_id, attr_id))
    
    all_adj_ids.append(adj_ids)

df["adj_attrsubj"] = all_adj_ids

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 737/737 [00:00<00:00, 22797.82it/s]

modèles actuels
un paradigme courant, les modèles actuels ne parviennent pas à effectuer des calculs de manière satisfaisante. Pour y

agents conversationnels
à leur utilisation dans des agents conversationnels incarnés ou dans d’autres domaines de recherche comme la linguistique, où

modèle capable (attr)
Nous avons démontré que notre modèle est capable de générer des gestes convaincants en surpassant l’état de l’art

modèles français
stéréotypes se reflètent dans les modèles de langue français. Nous adaptons le jeu de données StereoSet à la langue

modèles français
les biais constatés dans les modèles français. De plus, nous observons que l’utilisation de corpus multilingues pour

modèles grands
des amorces à plusieurs grands modèles de langue, et obtenons un score F1 compris entre 56,6 et 71,8 avec

modèle meilleur
relativement petite pour notre meilleur modèle. Cela suggère qu’avoir le patient en tant que point de départ du

modèles génératifs
des variables linguistiques et des mod

### Composé nominal modifiant une entité-LLM

In [37]:
all_role_nc = set()
all_role_nc_ids = []

for i, row in tqdm(df.iterrows(), total = df.shape[0]):
    llm_entities_ids = row["llm_entities_ids"]
    doc = row["analyzed_text"]

    role_nc_ids = []
    
    for e_id in llm_entities_ids:
        tok = doc[e_id]
        deprel = tok.dep_
        children = tok.children
        
        for child in children:
            if child.dep_ == "nmod" and child.pos_ in ["NOUN", "ADJ"] and list(child.children) == []:
                all_role_nc.add(child.lemma_)
                print(tok, child.orth_)
                print(doc[max(0, e_id-5):min(len(doc), e_id+15)])
                print()

                n_id = get_token_idx(child, doc)
                role_nc_ids.append((e_id, n_id))

    all_role_nc_ids.append(role_nc_ids)

df["noun_compound"] = all_role_nc_ids

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 737/737 [00:00<00:00, 38766.99it/s]


modèle -
être utilisées pour entraîner un modèle de question-réponse sur un nouveau corpus d’archives numérisées.

modèle réponse
être utilisées pour entraîner un modèle de question-réponse sur un nouveau corpus d’archives numérisées.

LLM -
résultats montrent que seuls les LLM fine-tunés atteignent la précision du modèle de référence. L’analyse des

modèle vanille
améliore de manière significative le modèle BERT vanille de référence. Nous démontrons empiriquement l’efficacité de notre approche par

modèle Base
d’évaluation FLEURS-102. Notre modèle HuBERT Base obtient des résultats similaires face à l’approche multilingue w2v-bert

Llama 8B.
décodeurs causaux, jusqu’à Llama 3 8B.

modèles GPT-2
générées artificiellement en adaptant des modèles GPT-2. Les résultats montrent que les données artificielles ne sont pas encore suffisamment

Mistral Moderation
, dont GPT-4o mini et Mistral Moderation , sur des requêtes multilingues dans des contextes variés. Les résultats montrent

Claude Op

### Compléments du nom associés à une entité-LLM

In [38]:
all_gen_nc = set()
all_gen_nc_ids = []

for i, row in tqdm(df.iterrows(), total = df.shape[0]):
    llm_entities_ids = row["llm_entities_ids"]
    doc = row["analyzed_text"]

    gen_nc_ids = []
    
    for e_id in llm_entities_ids:
        tok = doc[e_id]
        deprel = tok.dep_
        head = tok.head
        children = tok.children

        if deprel == "nmod" and head.pos_ == "NOUN":
            case_children = [child.lemma_ for child in children if child.dep_ == "case" and "d" in child.orth_]
            
            if any(case_children):
                all_gen_nc.add(head.lemma_)
                print(head.lemma_, case_children, tok)
                print(doc[max(0, e_id-5):min(len(doc), e_id+15)])
                print()

                head_id = get_token_idx(head, doc)
                gen_nc_ids.append((head_id, e_id))
    
    all_gen_nc_ids.append(gen_nc_ids)        

df["compl_noms"] = all_gen_nc_ids

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 737/737 [00:00<00:00, 21739.63it/s]

intégration ['dans'] modèles
du raisonnement numérique dans les modèles de langue : État de l’art et direction de recherche.
Ces

intégration ['dans'] modèles
du raisonnement numérique dans les modèles de langue a suscité un intérêt grandissant. Pourtant, bien que l’entraînement

entraînement ['de'] modèles
bien que l’entraînement des modèles de langue sur des données numériques soit devenu un paradigme courant, les modèles

travail ['dans'] modèles
le raisonnement numérique dans les modèles de langue et dans un second temps nous discutons des différentes perspectives de recherche

compétence ['de'] modèles
augmenter les compétences numériques des modèles.

utilisation ['dans'] agents
à leur utilisation dans des agents conversationnels incarnés ou dans d’autres domaines de recherche comme la linguistique, où

représentation ['dans'] modèles
représentations contextuelles stéréotypées dans les modèles de langue français : mieux les identifier pour ne pas les reproduire.


capacité ['de']

In [39]:
# Sauvegarde des modifications apportées aux données

# with open(os.path.join(data_folder, processed_jeptalnrecital_file_path) + ".pkl", "wb") as f:
#     pickle.dump(df, f)

# with open(os.path.join(data_folder, processed_jeptalnrecital_file_path) + ".pkl", "rb") as f:
#     df = pickle.load(f)